In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [2]:
from sqlalchemy import create_engine

engine = create_engine(
    "postgresql+psycopg2://harikasunkara@localhost:5432/telco"
)

df = pd.read_sql("SELECT * FROM customers", engine)

df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.shape

(7043, 21)

In [4]:
X = df.drop(columns=["customerid", "churn"])   # features which is not necessary for this 
y = df["churn"].map({"No": 0, "Yes": 1})   # target

In [5]:
X.shape

(7043, 19)

In [6]:
y.value_counts()

churn
0    5174
1    1869
Name: count, dtype: int64

In [7]:
y.value_counts(normalize=True)*100


churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64

In [8]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   seniorcitizen     7043 non-null   int64  
 2   partner           7043 non-null   object 
 3   dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   phoneservice      7043 non-null   object 
 6   multiplelines     7043 non-null   object 
 7   internetservice   7043 non-null   object 
 8   onlinesecurity    7043 non-null   object 
 9   onlinebackup      7043 non-null   object 
 10  deviceprotection  7043 non-null   object 
 11  techsupport       7043 non-null   object 
 12  streamingtv       7043 non-null   object 
 13  streamingmovies   7043 non-null   object 
 14  contract          7043 non-null   object 
 15  paperlessbilling  7043 non-null   object 
 16  paymentmethod     7043 non-null   object 


In [9]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['seniorcitizen', 'tenure', 'monthlycharges', 'totalcharges']
Categorical: ['gender', 'partner', 'dependents', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod']


### SPLIT THE DATA

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
print(X_train.shape)
print(X_test.shape)

(5634, 19)
(1409, 19)


### preprocessing

### Numeric columns should be scaled, and categorical columns should be one-hot encoded:

In [12]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## train Logistic Regression on the processed data.

In [14]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [15]:
logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['seniorcitizen', 'tenure',
                                                   'monthlycharges',
                                                   'totalcharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'partner',
                                                   'dependents', 'phoneservice',
                                                   'multiplelines',
                                                   'internetservice',
                                                   'onlinesecurity',
                                                   'onlinebackup',
                                                   'deviceprotection',
                                                   'techsupport', 'streamingtv',
                                                   'streamingmovies',
                                                   'contract',
                                                   'paperlessbilling',
                                                   'paymentmethod'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [16]:
logistic_model.predict(X_test)

array([0, 0, 0, ..., 0, 0, 0])

In [17]:
y_pred = logistic_model.predict(X_test)

In [18]:
y_prob = logistic_model.predict_proba(X_test)[:, 1]

In [19]:
y_prob

array([0.02050837, 0.00470985, 0.0377156 , ..., 0.38990983, 0.00418928,
       0.00624496])

In [20]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.801277501774308
Precision: 0.6577181208053692
Recall: 0.5240641711229946
F1 Score: 0.5833333333333334
ROC-AUC: 0.8370701903949984


In [21]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409



In [22]:
confusion_matrix(y_test, y_pred)

array([[933, 102],
       [178, 196]])

In [23]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

random_forest_model.fit(X_train, y_train)

rf_pred = random_forest_model.predict(X_test)
rf_prob = random_forest_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

Accuracy: 0.7785663591199432
Precision: 0.6115107913669064
Recall: 0.45454545454545453
F1 Score: 0.5214723926380368
ROC-AUC: 0.8102172621354207


In [24]:
from xgboost import XGBClassifier

In [25]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

In [26]:
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred))
print("Recall:", recall_score(y_test, xgb_pred))
print("F1 Score:", f1_score(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))

Accuracy: 0.7963094393186657
Precision: 0.6505190311418685
Recall: 0.5026737967914439
F1 Score: 0.5671191553544495
ROC-AUC: 0.8394404402077038


In [27]:
import numpy as np

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

for threshold in thresholds:
    pred_threshold = (y_prob >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Precision:", round(precision_score(y_test, pred_threshold), 3))
    print("Recall:", round(recall_score(y_test, pred_threshold), 3))
    print("F1:", round(f1_score(y_test, pred_threshold), 3))
    print()

Threshold: 0.5
Precision: 0.658
Recall: 0.524
F1: 0.583

Threshold: 0.45
Precision: 0.605
Recall: 0.591
F1: 0.598

Threshold: 0.4
Precision: 0.573
Recall: 0.639
F1: 0.604

Threshold: 0.35
Precision: 0.554
Recall: 0.684
F1: 0.612

Threshold: 0.3
Precision: 0.53
Recall: 0.741
F1: 0.618



In [28]:
final_threshold = 0.30

final_pred = (y_prob >= final_threshold).astype(int)

In [29]:
confusion_matrix(y_test, final_pred)

array([[789, 246],
       [ 97, 277]])

In [30]:
print(classification_report(y_test, final_pred))

              precision    recall  f1-score   support

           0       0.89      0.76      0.82      1035
           1       0.53      0.74      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409



“I lowered the classification threshold from 0.50 to 0.30 to improve churn recall. This increased recall from about 52% to 74%, reducing false negatives substantially. Although precision decreased, this tradeoff was acceptable for a retention use case where missing a likely churner is more costly than contacting some customers who may stay.”

In [31]:
X_all = df.drop(columns=["customerid", "churn"])

In [32]:
all_prob = logistic_model.predict_proba(X_all)[:, 1]

In [33]:
results = df.copy()

results["churn_probability"] = all_prob

In [34]:
results["risk_level"] = pd.cut(
    results["churn_probability"],
    bins=[0, 0.30, 0.60, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

In [35]:
results[
    ["customerid", "churn_probability", "risk_level"]
].head(10)

,customerid,churn_probability,risk_level
0,7590-VHVEG,0.620557,High
1,5575-GNVDE,0.043771,Low
2,3668-QPYBK,0.297236,Low
3,7795-CFOCW,0.028802,Low
4,9237-HQITU,0.691593,High
5,9305-CDSKC,0.789599,High
6,1452-KIOVK,0.463646,Medium
7,6713-OKOMC,0.305953,Medium
8,7892-POOKP,0.623476,High
9,6388-TABGU,0.009541,Low


In [36]:
results["risk_level"].value_counts()

risk_level
Low       4398
Medium    1610
High      1035
Name: count, dtype: int64

In [37]:
results.to_sql(
    "customer_churn_predictions",
    engine,
    if_exists="replace",
    index=False
)

43

In [38]:
pd.read_sql(
    """
    SELECT
        risk_level,
        COUNT(*) AS customers
    FROM customer_churn_predictions
    GROUP BY risk_level
    ORDER BY customers DESC;
    """,
    engine
)

,risk_level,customers
0,Low,4398
1,Medium,1610
2,High,1035


In [39]:
results["revenue_at_risk"] = np.where(
    results["risk_level"] == "High",
    results["monthlycharges"],
    0
)

In [40]:
results["revenue_at_risk"].sum()

85104.65000000001

In [41]:
results.to_sql(
    "customer_churn_predictions",
    engine,
    if_exists="replace",
    index=False
)

43

In [42]:
pd.read_sql("SELECT * FROM customer_churn_predictions", engine)

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn,churn_probability,risk_level,revenue_at_risk
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0.620557,High,29.85
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,1889.50,No,0.043771,Low,0.00
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0.297236,Low,0.00
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0.028802,Low,0.00
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0.691593,High,70.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,One year,Yes,Mailed check,84.80,1990.50,No,0.112863,Low,0.00
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No,0.124919,Low,0.00
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,Month-to-month,Yes,Electronic check,29.60,346.45,No,0.390946,Medium,0.00
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes,0.732787,High,74.40


In [43]:
results.to_csv(
    "../data/customer_churn_predictions.csv",
    index=False
)

In [44]:
import joblib

joblib.dump(
    logistic_model,
    "../models/logistic_churn_model.pkl"
)

print("Model saved successfully")

Model saved successfully


In [45]:
final_threshold = 0.30

joblib.dump(
    final_threshold,
    "../models/churn_threshold.pkl"
)

['../models/churn_threshold.pkl']